In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)


In [3]:
import os
os.getcwd()


'c:\\Users\\saksh\\OneDrive\\Documents\\GitHub\\Aadhar-card-dataAnalysis-UAIDI-Hackathon-project\\Notebook'

In [5]:
import os
os.listdir("..")


['.git',
 '.gitattributes',
 '.gitignore',
 'Data',
 'LICENSE',
 'Notebook',
 'README.md']

In [6]:
os.listdir("../Data")


['Preprocessed_Data', 'Raw_Data']

In [7]:
os.listdir("../data/Preprocessed_Data")


['clean_biometric_base.csv',
 'clean_demographic_base.csv',
 'clean_enrolment_base.csv']

In [ ]:
DATA_PATH = "../Data/Preprocessed_Data"

enrol_df = pd.read_csv(f"{DATA_PATH}/clean_enrolment_base.csv", parse_dates=["date"])
demo_df  = pd.read_csv(f"{DATA_PATH}/clean_demographic_base.csv", parse_dates=["date"])
bio_df   = pd.read_csv(f"{DATA_PATH}/clean_biometric_base.csv", parse_dates=["date"])


In [9]:
def add_temporal_features(df):
    df = df.copy()
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["day_of_week"] = df["date"].dt.dayofweek
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
    return df


In [10]:
enrol_feat = add_temporal_features(enrol_df)

enrol_feat["total_enrolments"] = (
    enrol_feat["age_0_5"] +
    enrol_feat["age_5_17"] +
    enrol_feat["age_18_greater"]
)

enrol_feat["child_enrolments"] = (
    enrol_feat["age_0_5"] +
    enrol_feat["age_5_17"]
)

enrol_feat["adult_ratio"] = np.where(
    enrol_feat["total_enrolments"] > 0,
    enrol_feat["age_18_greater"] / enrol_feat["total_enrolments"],
    0
)

enrol_feat["child_share"] = np.where(
    enrol_feat["total_enrolments"] > 0,
    enrol_feat["child_enrolments"] / enrol_feat["total_enrolments"],
    0
)


In [11]:
demo_feat = add_temporal_features(demo_df)

demo_feat["total_demographic_updates"] = (
    demo_feat["demo_age_5_17"] +
    demo_feat["demo_age_17_"]
)


In [12]:
bio_feat = add_temporal_features(bio_df)

bio_feat["total_biometric_updates"] = (
    bio_feat["bio_age_5_17"] +
    bio_feat["bio_age_17_"]
)


In [13]:
join_keys = ["date", "state_clean", "district_clean", "pincode"]

demo_bio_combined = pd.merge(
    demo_feat,
    bio_feat,
    on=join_keys,
    how="inner",
    suffixes=("_demo", "_bio")
)


In [14]:
demo_bio_combined["biometric_to_demographic_ratio"] = np.where(
    demo_bio_combined["total_demographic_updates"] > 0,
    demo_bio_combined["total_biometric_updates"] /
    demo_bio_combined["total_demographic_updates"],
    0
)


In [15]:
enrol_feat.to_csv("feature_enrolment.csv", index=False)
demo_feat.to_csv("feature_demographic.csv", index=False)
bio_feat.to_csv("feature_biometric.csv", index=False)
demo_bio_combined.to_csv("feature_demo_bio_combined.csv", index=False)


In [16]:
import os

FEATURE_PATH = "../Data/Feature_Data"
os.makedirs(FEATURE_PATH, exist_ok=True)


In [17]:
enrol_feat.to_csv(f"{FEATURE_PATH}/feature_enrolment.csv", index=False)
demo_feat.to_csv(f"{FEATURE_PATH}/feature_demographic.csv", index=False)
bio_feat.to_csv(f"{FEATURE_PATH}/feature_biometric.csv", index=False)
demo_bio_combined.to_csv(
    f"{FEATURE_PATH}/feature_demo_bio_combined.csv",
    index=False
)
